In [ ]:
# general imports
import os
import scipy.stats as stats
import itertools
from itertools import product, combinations_with_replacement
import pickle
import pandas as pd
import json
import re
import numpy as np
import glob
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc
import warnings
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

# local imports
from utils.unicode_utils import apply_greek_labels_heatmap, introduce_greek_letters, replace_greek_letters, apply_greek_labels
from utils.plotting_utils import cm_to_inches, setup_plotting_environment, plot_overview_labels_along_y_axis, create_sankey_diagram_strength
def get_common_index(series1, series2):
    series1 = series1.copy().dropna()
    series2 = series2.copy().dropna()
    common_index = series1.index.intersection(series2.index)
    return series1.loc[common_index], series2.loc[common_index]

setup_plotting_environment()

# ------- CRITICAL -------
alt_flag = False # use an alternative cell type set (more fine-grained)
from utils.color_constants import CELL_TYPE_COLORS, ALL_CELL_TYPES

# set up folders
input_data_folder = f"../../data/pilot/"
os.makedirs(input_data_folder, exist_ok=True)

output_folder = f"../../outputs/pilot/"
os.makedirs(output_folder, exist_ok=True)

base_plot_folder = os.path.join(output_folder, "plots")
os.makedirs(base_plot_folder, exist_ok=True)
intermediate_output_folder = os.path.join(output_folder, "intermediate_outputs")
os.makedirs(intermediate_output_folder, exist_ok=True)
output_tables_folder = os.path.join(output_folder, "output_tables")
os.makedirs(output_tables_folder, exist_ok=True)

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
gene_to_cytokine = cytokine_info.set_index("gene")['gene_to_cytokine'].dropna().to_dict()
cytokine_to_gene = {v: k for k, v in gene_to_cytokine.items()}
ALL_CYTOKINES_ORDER = cytokine_info['gene_to_cytokine'].dropna().tolist()

cytokine_name_to_level_1_family = cytokine_info.reset_index().set_index("name")["family_level_1"].to_dict()
cytokine_name_to_level_2_family = cytokine_info.reset_index().set_index("name")["family_level_2"].to_dict()
cytokine_name_to_functional = cytokine_info.reset_index().set_index("name")["functional_grouping"].to_dict()

# gene-level names (e.g. IL-32, C3, C5, LT-alpha-beta) are aliases of cytokine_info names
gene_level_alias = {alias: name for name, alias in cytokine_info["gene_to_cytokine"].dropna().items() if alias != name}
for name_map in [cytokine_name_to_level_1_family, cytokine_name_to_level_2_family, cytokine_name_to_functional]:
    name_map.update({alias: name_map[name] for alias, name in gene_level_alias.items()})

cytokine_level_1_order = list(dict.fromkeys(cytokine_name_to_level_1_family.values()))
cytokine_level_2_order = list(dict.fromkeys(cytokine_name_to_level_2_family.values()))
level2_to_level1 = cytokine_info.set_index("family_level_2")["family_level_1"].to_dict()

# tested subset
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]

# ordering
cytokine_info_order = pd.read_excel("../../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

# Adata

one donor

15 cytokines + Ctrl, at 5 / 25 / 100 ng, harvested at 5h and 24h

2 wells per condition (6 for Ctrl), roughly 1,000–2,800 cells per condition across all cell types

This pilot screen had some technical issues (see the low corr. for IL1B, TGF-beta, or TNF-alpha, for example).

It should only be used where concordance at 24h with the main screen results establish a useful perturbation.

In [ ]:
adata = sc.read_h5ad(os.path.join(input_data_folder, "10M_pilot_exp_20260429_pipeline_re-run_final.h5ad"))
adata.obs = adata.obs.astype(str)
adata = adata[adata.obs["final_cell_type"].notna()]
adata = adata[adata.obs['cytokine'].notna()]

In [ ]:
from utils.cell_type_mapping import get_relevant_cell_types
relevant_cell_types, relevant_cell_type_conversion, relevant_cell_to_pseudobulk = get_relevant_cell_types(alt_flag)

cytokine_conversion = {
    "IL1b": "IL-1-beta",
    "IL10": "IL-10",
    "41BBL": "4-1BBL",
    "C5a": "C5a",
    "Ctrl": "Ctrl",
    "GMCSF": "GM-CSF",
    "IFNb": "IFN-beta",
    "IFNg": "IFN-gamma",
    'IL12': 'IL-12',
    'IL21': 'IL-21',
    'IL2': 'IL-2',
    'IL4': 'IL-4',
    'IL5': 'IL-5',
    'SCF': 'SCF',
    'TGFb1-inhibitor': 'Lap',
    'TNFa': 'TNF-alpha',
}

missing_cell_types = set(adata.obs["final_cell_type"].unique()) - set(relevant_cell_type_conversion.keys())
missing_cytokines = set(adata.obs["cytokine"].unique()) - set(cytokine_conversion.keys())

print("Missing cell type mappings:", sorted(missing_cell_types))
print("Missing cytokine mappings:", sorted(missing_cytokines))

adata.obs['final_cell_type'] = adata.obs['final_cell_type'].astype(str).apply(lambda x: relevant_cell_type_conversion.get(x, x))
adata.obs['cytokine'] = adata.obs['cytokine'].astype(str).apply(lambda x: cytokine_conversion.get(x, x))
group_cols = ["final_cell_type", "time", "cytokine", "concentration"]
for col in group_cols:
    adata.obs[col] = pd.Categorical(adata.obs[col].astype(str))

## Pseudobulk

In [ ]:
group_cols = ["final_cell_type", "time", "cytokine", "concentration"]

# one replicate label per unique bc1_well within each condition
rep_map = adata.obs[group_cols + ["bc1_well"]].drop_duplicates().sort_values(group_cols + ["bc1_well"])
rep_map = rep_map.assign(replicate=lambda df: (df.groupby(group_cols, observed=True).cumcount().add(1)))
rep_map["replicate"] = "rep" + rep_map["replicate"].astype(str)

# add back to adata.obs
adata.obs = adata.obs.merge(rep_map, on=group_cols + ["bc1_well"], how="left")

In [ ]:
# counts per condition (this one is pretty shallow)
adata.obs.groupby(["time", "cytokine", "concentration", "replicate"], observed=True).size().median()

In [ ]:
group_cols = ["final_cell_type", "time", "cytokine", "concentration", "replicate"]

# pseudobulk
pseudobulk = sc.get.aggregate(adata, by=group_cols, layer='raw_counts', func="sum")
pseudobulk.X = pseudobulk.layers["sum"].copy()

# add cell count to pseudobulk obs
all_counts_by_condition = adata.obs.groupby(group_cols, observed=True).size()
all_counts_by_condition_flat = all_counts_by_condition.copy()
all_counts_by_condition_flat.index = all_counts_by_condition_flat.index.map(lambda x: "_".join(x))
pseudobulk.obs['cell_count'] = all_counts_by_condition_flat

# add count_nonzero to pseudobulk layers
count_nonzero = sc.get.aggregate(adata, by=group_cols, layer='raw_counts', func="count_nonzero")
pseudobulk.layers['count_nonzero'] = count_nonzero.layers['count_nonzero'].copy()

# add percent nonzero to pseudobulk layers
cell_count_arr = pseudobulk.obs["cell_count"].to_numpy()[:, None]
pseudobulk.layers["pct_nonzero"] = (pseudobulk.layers["count_nonzero"] / cell_count_arr * 100)

# Pseudobulk filtering

In [ ]:
# min cell filtering
min_cells = 10
pseudobulk = pseudobulk[pseudobulk.obs["cell_count"] >= min_cells].copy()

# max filter
questionable_genes = set(pd.read_csv(os.path.join("../../data/main_screen", "questionable_genes.csv"))['gene'])
pseudobulk = pseudobulk[:, ~pseudobulk.var_names.isin(questionable_genes)].copy()

# filter to protein coding genes
with open(os.path.join(f"../../annotations/", "protein_coding_genes.txt"), "r") as f:
    protein_coding_genes = [line.strip() for line in f.readlines()]
pseudobulk = pseudobulk[:, pseudobulk.var_names.isin(protein_coding_genes)].copy()

# EdgeR

In [ ]:
run_pertpy = True
if run_pertpy:
    from utils.edger_utils import run_de_by_strata

In [ ]:
def cytokine_case_selector(obs, stratum):
    return (
        (obs["final_cell_type"].astype(str).values == str(stratum["final_cell_type"]))
        & (obs["time"].astype(str).values == str(stratum["time"]))
        & (obs["concentration"].astype(str).values == str(stratum["concentration"]))
        & (obs["cytokine"].astype(str).values != "Ctrl")
    )

def cytokine_ctrl_reference_selector(obs, stratum, contrast_col=None, baseline="Ctrl"):
    return (
        (obs["final_cell_type"].astype(str).values == str(stratum["final_cell_type"]))
        & (obs["time"].astype(str).values == str(stratum["time"]))
        & (obs["concentration"].astype(str).values == str(baseline))
        & (obs["cytokine"].astype(str).values == str(baseline))
    )

if run_pertpy:
    run_de_by_strata(
        pseudobulk=pseudobulk,
        strata_cols=["final_cell_type", "time", "concentration"],
        contrast_col="cytokine",
        baseline="Ctrl",
        out_dir=os.path.join(input_data_folder, "DE_results_cytokine"),
        filename_prefix="cytokine_DE",
        min_cells=10,
        min_samples_detected=2,
        min_count=5,
        case_selector=cytokine_case_selector,
        reference_selector=cytokine_ctrl_reference_selector,
    )
    run_de_by_strata(
        pseudobulk=pseudobulk,
        strata_cols=["final_cell_type", "cytokine", "concentration"],
        contrast_col="time",
        baseline="5hr",
        out_dir=os.path.join(input_data_folder, "DE_results_time"),
        filename_prefix="time_DE",
        min_cells=10,
        min_samples_detected=2,
        min_count=5,
    )

In [ ]:
def merge_de_result_files(
    results_dir,
    output_filename="all_DE_results.parquet",
    pattern="*.parquet",
    delete_inputs=True,
    filename_parser=None,
):
    """
    Merge per-stratum DE result parquet files into one parquet file.

    Parameters
    ----------
    results_dir : str
        Directory containing per-comparison parquet files.
    output_filename : str
        Name of merged output parquet.
    pattern : str
        Glob pattern for input files.
    delete_inputs : bool
        Delete individual files after successful merge.
    filename_parser : callable or None
        Optional function taking filepath -> dict of metadata columns.
        Used only for columns missing from the file itself.
    """
    output_path = os.path.join(results_dir, output_filename)

    result_files = sorted(glob.glob(os.path.join(results_dir, pattern)))
    result_files = [f for f in result_files if os.path.abspath(f) != os.path.abspath(output_path)]

    if len(result_files) == 0:
        raise FileNotFoundError(f"No parquet files found in {results_dir} matching {pattern}")

    dfs = []

    for f in result_files:
        df = pd.read_parquet(f)

        # Add source file for debugging/provenance
        df["source_file"] = os.path.basename(f)

        # Optional fallback metadata from filename
        if filename_parser is not None:
            meta = filename_parser(f)
            for key, val in meta.items():
                if key not in df.columns:
                    df[key] = val

        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)

    # Write first
    tmp_output_path = output_path + ".tmp"
    merged.to_parquet(tmp_output_path, index=False)

    # Sanity-check written file
    check = pd.read_parquet(tmp_output_path)
    if len(check) != len(merged):
        raise RuntimeError("Merged file row count changed after writing; not deleting inputs.")

    os.replace(tmp_output_path, output_path)

    if delete_inputs:
        for f in result_files:
            os.remove(f)

    return merged

if run_pertpy:
    _ = merge_de_result_files(
        results_dir=os.path.join(input_data_folder, "DE_results_cytokine"),
        output_filename="all_DE_results_cytokine.parquet",
        pattern="*.parquet",
        delete_inputs=True,
    )

In [ ]:
edger_results_time = pd.read_parquet(os.path.join(input_data_folder, "DE_results_time", "all_DE_results_time.parquet"))
edger_results = pd.read_parquet(os.path.join(input_data_folder, "DE_results_cytokine", "all_DE_results_cytokine.parquet"))

In [ ]:
RELEVANT_CELL_TYPES = all_counts_by_condition.groupby(level=0, observed=True).median()
RELEVANT_CELL_TYPES = RELEVANT_CELL_TYPES[RELEVANT_CELL_TYPES > 30].index.tolist()
pseudobulk_df_sum = pd.DataFrame(
    pseudobulk.layers["sum"],
    index=pd.MultiIndex.from_frame(pseudobulk.obs[["final_cell_type", "time", "cytokine", "concentration", "replicate"]]),
    columns=pseudobulk.var_names
)
pseudobulk_df_sum = pseudobulk_df_sum.T
pseudobulk_df_sum.columns = pd.MultiIndex.from_frame(pseudobulk_df_sum.columns.to_frame(index=False).astype(str))

pct_nonzero_df = pd.DataFrame(
    pseudobulk.layers["count_nonzero"],
    index=pd.MultiIndex.from_frame(pseudobulk.obs[["final_cell_type", "time", "cytokine", "concentration", "replicate"]]),
    columns=pseudobulk.var_names
)
pct_nonzero_df = pct_nonzero_df.T
pct_nonzero_df.columns = pd.MultiIndex.from_frame(
    pct_nonzero_df.columns.to_frame(index=False).astype(str)
)

In [ ]:
from utils.processing_utils import normalize_df_to_rpm

# aggregate across all cell types
pseudobulk_df_sum_all = pseudobulk_df_sum.T.groupby(level = [1, 2, 3, 4]).sum().T
pseudobulk_df_sum_all_norm = normalize_df_to_rpm(pseudobulk_df_sum_all, along_axis=0)

# normalize the rest to rpm
pseudobulk_df_sum = pseudobulk_df_sum[RELEVANT_CELL_TYPES]
pseudobulk_df_sum_norm = normalize_df_to_rpm(pseudobulk_df_sum, along_axis=0)
pseudobulk_df_sum_norm = pseudobulk_df_sum_norm.dropna(how="all", axis=1)

In [ ]:
unfiltered_genes = pseudobulk_df_sum_norm.index
inclusive_filters_by_cell_type = {}
for cell_type in RELEVANT_CELL_TYPES:
    df = pseudobulk_df_sum[cell_type]
    df = df[df.max(axis=1) > 20]
    df = df[~df.index.isin(questionable_genes)]
    print(cell_type, len(df))
    inclusive_filters_by_cell_type[cell_type] = df.index

## Compare fold changes

In [ ]:
# load vals from the main screen
raw_log2fcs_main = pd.read_parquet("../../outputs/main_screen/output_tables/raw_log2fc_values.parquet")
allowed_genes_main = raw_log2fcs_main.index.intersection(protein_coding_genes)
raw_log2fcs_main = raw_log2fcs_main.loc[allowed_genes_main]

In [ ]:
edger_results['protein_coding'] = edger_results['gene'].isin(protein_coding_genes)
edger_results = edger_results[edger_results['protein_coding']].copy()

raw_log2fcs_pilot = edger_results.set_index(['gene', 'cytokine', 'final_cell_type', 'time', 'concentration'])['manual_log_fc'].unstack(
    ['final_cell_type', 'cytokine', 'time', 'concentration'])
raw_log2fcs_pilot_100ng = raw_log2fcs_pilot.xs('100ng', level='concentration', axis=1)

In [ ]:
shared_cytokines = set(raw_log2fcs_pilot_100ng.columns.get_level_values('cytokine')).intersection(raw_log2fcs_main.columns.get_level_values('cytokine'))
shared_cytokines = sorted(shared_cytokines)

raw_log2fcs_main = raw_log2fcs_main[RELEVANT_CELL_TYPES].copy()
raw_log2fcs_main = raw_log2fcs_main.loc[:, raw_log2fcs_main.columns.get_level_values('cytokine').isin(shared_cytokines)].copy()
raw_log2fcs_pilot_100ng = raw_log2fcs_pilot_100ng[RELEVANT_CELL_TYPES].copy()
raw_log2fcs_pilot_100ng = raw_log2fcs_pilot_100ng.loc[:, raw_log2fcs_pilot_100ng.columns.get_level_values('cytokine').isin(shared_cytokines)].copy()

In [ ]:
cell_types = ["CD14 Mono", 'NK CD56low']
cytokines = ["IFN-beta", "IFN-gamma", "IL-12"]
fig, axes = plt.subplots(len(cell_types), len(cytokines), figsize=cm_to_inches((6, 4.5)), sharex=True, sharey=True)
axes = axes.flatten()

plot_i = 0
for cell_type in cell_types:
    for cytokine in cytokines:
        ax = axes[plot_i]
        values_main = raw_log2fcs_main.loc[allowed_genes_main, cell_type][cytokine].dropna()
        values_pilot = raw_log2fcs_pilot_100ng.loc[:, (cell_type, cytokine, "24hr")].dropna()
        common_index = values_main.index.intersection(values_pilot.index)

        inclusive_filter = inclusive_filters_by_cell_type[cell_type]
        common_index = [c for c in common_index if c in inclusive_filter]

        ax.scatter(values_pilot.loc[common_index], values_main.loc[common_index], s=3, alpha=0.5,
            edgecolors="none", rasterized=True, color=CELL_TYPE_COLORS[cell_type])
        r = np.corrcoef(values_pilot.loc[common_index], values_main.loc[common_index])[0, 1]
        ax.text(0.05, 0.95, f"r={r:.2f}", transform=ax.transAxes, verticalalignment="top")
        if cell_type == cell_types[0]:
            ax.set_title(introduce_greek_letters(f"{cytokine}"))
        if plot_i == 4:
            ax.set_xlabel("log2FC(pilot, 24h)")
        if plot_i == 3:
            ax.set_ylabel("log2FC(main screen)")
        ax.tick_params(length=2, axis='both')
        plot_i += 1

plt.tight_layout()
plt.savefig(os.path.join(base_plot_folder, f"pilot_vs_main_scatter.svg"), dpi=400)
plt.show()

In [ ]:
cell_types = ["CD14 Mono", 'NK CD56low']
cytokines = ["IFN-beta", "IFN-gamma", "IL-12"]
fig, axes = plt.subplots(len(cell_types), len(cytokines), figsize=cm_to_inches((6, 4.5)), sharex=True, sharey=True)
axes = axes.flatten()

plot_i = 0
for cell_type in cell_types:
    for cytokine in cytokines:
        ax = axes[plot_i]

        values_pilot_24h = raw_log2fcs_pilot_100ng.loc[:, (cell_type, cytokine, "24hr")].dropna()
        values_pilot_5h = raw_log2fcs_pilot_100ng.loc[:, (cell_type, cytokine, "5hr")].dropna()
        common_index = values_pilot_24h.index.intersection(values_pilot_5h.index)
        ax.scatter(values_pilot_24h.loc[common_index], values_pilot_5h.loc[common_index], s=3, alpha=0.5,
            edgecolors="none", rasterized=True, color=CELL_TYPE_COLORS[cell_type])
        r = np.corrcoef(values_pilot_24h.loc[common_index], values_pilot_5h.loc[common_index])[0, 1]
        ax.text(0.05, 0.95, f"r={r:.2f}", transform=ax.transAxes, verticalalignment="top")

        if cell_type == cell_types[0]:
            ax.set_title(introduce_greek_letters(f"{cytokine}"))
        if plot_i == 4:
            ax.set_xlabel("log2FC(pilot, 24 hours)")
        if plot_i == 3:
            ax.set_ylabel("log2FC(pilot, 5 hours)")
        ax.tick_params(length=2, axis='both')
        plot_i += 1

plt.tight_layout()
plt.savefig(os.path.join(base_plot_folder, f"pilot_24h_vs_5h_scatter.svg"), dpi=400)
plt.show()

In [ ]:
output_genes = ['IFNG']
cytokines = ['Ctrl', 'IL-12', 'IL-2']
times = ['5hr', '24hr']
concentration = '100ng'          # Ctrl has concentration 'Ctrl'

df = pseudobulk_df_sum_all_norm.loc[output_genes].T.reset_index()
df = df[df['cytokine'].isin(cytokines) & df['time'].isin(times)
        & ((df['concentration'] == concentration) | (df['cytokine'] == 'Ctrl'))]
df = df.melt(id_vars=['time', 'cytokine', 'concentration', 'replicate'],
             value_vars=output_genes, var_name='gene', value_name='cpm')
df['time'] = df['time'].str.replace('hr', ' h')
time_labels = [t.replace('hr', ' h') for t in times]
palette = {'5 h': 'tab:blue', '24 h': 'tab:red'}

fig, axes = plt.subplots(1, len(output_genes), figsize=cm_to_inches((4 * len(output_genes), 4)), squeeze=False)
for i, gene in enumerate(output_genes):
    ax = axes[0, i]
    sub = df[df['gene'] == gene]
    sns.barplot(data=sub, x='cytokine', y='cpm', hue='time', order=cytokines, hue_order=time_labels,
                palette=palette, errorbar=None, ax=ax, legend=(i == len(output_genes) - 1))
    sns.stripplot(data=sub, x='cytokine', y='cpm', hue='time', order=cytokines, hue_order=time_labels,
                  dodge=True, palette={t: 'black' for t in time_labels}, size=2, ax=ax, legend=False)
    ax.set_title(gene)
    ax.set_xlabel('')
    ax.set_ylabel('cpm (all cells)' if i == 0 else '')
    ax.set_xticks(range(len(cytokines)), [introduce_greek_letters(c) for c in cytokines], rotation=90)
    ax.spines[['top', 'right']].set_visible(False)
    
axes[0, -1].legend(title='', frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.savefig(os.path.join(base_plot_folder, "pilot_TNF_IFNG_IL-12_IL-2_by_time.svg"), bbox_inches='tight')
plt.show()